[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/umangbagadi03/build-your-own-coding-agent/blob/main/notebooks/05_first_real_agent.ipynb)

# 05. Your First Real Agent

**Part 1: The Loop** · about 25 minutes

This is the payoff. We'll point the agent at a small Python project with failing tests and let it find the bugs, fix the code and confirm the tests pass. You've already seen every piece. This notebook connects them to real files.

> **Offline mode is a scripted demonstration.** The script already knows this exact fix, so it shows you how the loop works without showing any real intelligence. **To see a model actually solve it, connect a real provider** (see notebook 01). Real runs vary, and a weak model may fail, which is instructive too.

**By the end you will have:**

- file tools that can't escape their folder
- a complete agent, with its size measured
- an independent check of whether it really fixed the bugs

## Safety first

A coding agent runs actions chosen by a model, so we keep this one on a short leash:

- It works only inside a **temporary folder** created below.
- Its file tools **refuse paths outside** that folder.
- It has **no shell tool**. The only command it can trigger is our fixed test command.
- That test command runs with a **minimal environment**, so your API keys aren't visible to it.

One limit to keep in mind: `run_tests` executes Python that the model may have written, and that code runs with your user permissions. This is not a true sandbox. Keep it to projects like this one until Part 2, or run it on Colab.

In [1]:
import pathlib, urllib.request

if not pathlib.Path("llm_setup.py").exists():  # Colab opens one notebook at a time, so fetch the helper
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/umangbagadi03/build-your-own-coding-agent/main/notebooks/llm_setup.py", "llm_setup.py"
    )

from llm_setup import connect

llm = connect()

OFFLINE MODE: a scripted local server, not a real model.
The HTTP calls are real. The answers are canned for these lessons.
Token counts are rough estimates and timings are simulated.
Set GROQ_API_KEY (free tier) or LLM_PROVIDER=ollama to use a real model.


In [2]:
import difflib
import inspect
import json
import os
import shutil
import subprocess
import sys
import tempfile
import time
from pathlib import Path

import requests

url = f"{llm.base_url}/chat/completions"

headers = {"Content-Type": "application/json"}
if llm.api_key:
    headers["Authorization"] = f"Bearer {llm.api_key}"

## 1. A project with two bugs

Here's a tiny shopping cart. `subtotal` ignores quantity, and `shipping` gets the free-shipping boundary wrong. The tests describe the correct behaviour.

In [3]:
BUGGY_CART = '''\
def subtotal(items):
    """Total price of all items, taking quantity into account."""
    return sum(item["price"] for item in items)


def shipping(total):
    """Shipping costs 5, and is free for orders of 50 or more."""
    return 0 if total > 50 else 5


def checkout_total(items):
    total = subtotal(items)
    return total + shipping(total)
'''

TESTS = '''\
import unittest

from cart import checkout_total, shipping, subtotal


class CartTests(unittest.TestCase):
    def test_subtotal_uses_quantity(self):
        items = [{"price": 10, "quantity": 2}, {"price": 5, "quantity": 1}]
        self.assertEqual(subtotal(items), 25)

    def test_free_shipping_at_exactly_50(self):
        self.assertEqual(shipping(50), 0)

    def test_shipping_below_50(self):
        self.assertEqual(shipping(49), 5)

    def test_checkout_total(self):
        self.assertEqual(checkout_total([{"price": 20, "quantity": 3}]), 60)


if __name__ == "__main__":
    unittest.main()
'''

workspace = Path(tempfile.mkdtemp(prefix="agent-workspace-")).resolve()
(workspace / "cart.py").write_text(BUGGY_CART)
(workspace / "test_cart.py").write_text(TESTS)

print("Files:", sorted(p.name for p in workspace.iterdir()))

Files: ['cart.py', 'test_cart.py']


## 2. The tools

An agent is only as capable, and only as dangerous, as its tools. There are four.

`safe_path` is the one to study. `resolve()` turns tricks like `../../secret` and symlinks into their real location, and then we check that it's still inside the workspace. **This limit lives in code, so nothing the model reads or writes can talk its way past it.**

In [4]:
def safe_path(path):
    """Turn a relative path into a real one, refusing anything outside the workspace."""
    full = (workspace / path).resolve()
    if not full.is_relative_to(workspace):
        raise PermissionError(f"{path!r} is outside the workspace")
    return full


def list_files():
    names = (p.relative_to(workspace).as_posix() for p in workspace.rglob("*") if p.is_file())
    return json.dumps(sorted(n for n in names if "__pycache__" not in n))


def read_file(path):
    return safe_path(path).read_text()


def write_file(path, content):
    target = safe_path(path)
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(content)
    return f"Wrote {len(content)} characters to {path}"


def run_tests():
    """Run the project's tests with a time limit and without our environment variables."""
    # Pass on only what Python needs to start (SYSTEMROOT is for Windows), so API keys stay out.
    env = {name: os.environ[name] for name in ("PATH", "SYSTEMROOT") if name in os.environ}
    try:
        done = subprocess.run(
            [sys.executable, "-B", "-m", "unittest", "discover"],
            cwd=workspace,
            env=env,
            capture_output=True,
            text=True,
            timeout=30,
        )
    except subprocess.TimeoutExpired:
        return "FAILED: the tests took longer than 30 seconds."
    output = done.stderr.replace(str(workspace), ".")[-2000:]  # keep only the tail, as output can be huge
    return ("PASSED\n" if done.returncode == 0 else "FAILED\n") + output

In [5]:
TOOLS = [
    {"type": "function", "function": {
        "name": "list_files",
        "description": "List every file in the project.",
        "parameters": {"type": "object", "properties": {}},
    }},
    {"type": "function", "function": {
        "name": "read_file",
        "description": "Read a file. The path is relative to the project root.",
        "parameters": {
            "type": "object",
            "properties": {"path": {"type": "string"}},
            "required": ["path"],
        },
    }},
    {"type": "function", "function": {
        "name": "write_file",
        "description": "Replace a file's entire contents, so always include the complete new file.",
        "parameters": {
            "type": "object",
            "properties": {"path": {"type": "string"}, "content": {"type": "string"}},
            "required": ["path", "content"],
        },
    }},
    {"type": "function", "function": {
        "name": "run_tests",
        "description": "Run the project's unit tests. Output starts with PASSED or FAILED.",
        "parameters": {"type": "object", "properties": {}},
    }},
]

REGISTRY = {"list_files": list_files, "read_file": read_file, "write_file": write_file, "run_tests": run_tests}

Before giving these to a model, let's attack them ourselves. These checks don't involve a model at all, so they run the same in every mode.

In [6]:
for attempt in ["../outside.txt", "../../../etc/passwd", "/etc/passwd"]:
    try:
        safe_path(attempt)
        raise AssertionError(f"{attempt} should have been blocked")
    except PermissionError:
        print(f"blocked: {attempt}")

# A symlink inside the workspace that points outside must be blocked too.
try:
    (workspace / "sneaky").symlink_to(Path(tempfile.gettempdir()).resolve())
except OSError:
    print("skipped the symlink check: creating symlinks isn't allowed here (common on Windows)")
else:
    try:
        safe_path("sneaky/secret.txt")
        raise AssertionError("The symlink should have been blocked")
    except PermissionError:
        print("blocked: sneaky/secret.txt (via a symlink)")
    (workspace / "sneaky").unlink()

assert safe_path("cart.py") == workspace / "cart.py"
print("allowed: cart.py")
print("\nCheckpoint passed.")

blocked: ../outside.txt
blocked: ../../../etc/passwd
blocked: /etc/passwd
blocked: sneaky/secret.txt (via a symlink)
allowed: cart.py

Checkpoint passed.


Now let's confirm the project really is broken before the agent touches it. If we skip this, we can't prove the agent changed anything.

In [7]:
before = run_tests()
print(before)
assert before.startswith("FAILED")

FAILED
FF.F
FAIL: test_checkout_total (test_cart.CartTests.test_checkout_total)
----------------------------------------------------------------------
Traceback (most recent call last):
  File "./test_cart.py", line 18, in test_checkout_total
    self.assertEqual(checkout_total([{"price": 20, "quantity": 3}]), 60)
    ~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: 25 != 60

FAIL: test_free_shipping_at_exactly_50 (test_cart.CartTests.test_free_shipping_at_exactly_50)
----------------------------------------------------------------------
Traceback (most recent call last):
  File "./test_cart.py", line 12, in test_free_shipping_at_exactly_50
    self.assertEqual(shipping(50), 0)
    ~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^
AssertionError: 5 != 0

FAIL: test_subtotal_uses_quantity (test_cart.CartTests.test_subtotal_uses_quantity)
----------------------------------------------------------------------
Traceback (most recent call last):
  File "./test_cart.py", 

## 3. The agent

This is everything from notebooks 01 to 04, all in one place. The system prompt tells the agent how to work: look first, confirm with tests, and don't claim success without evidence.

In [8]:
SYSTEM_PROMPT = """You are a careful coding agent working in a small Python project.

Work in this order:
1. Look at the files and run the tests to see what is failing.
2. Read the code and work out the cause. The tests describe the correct behaviour, so don't edit them.
3. Fix the code with write_file.
4. Run the tests again. If they still fail, keep going.

Finish with a short summary. Only say the tests pass if you saw them pass."""


def call_model(messages):
    payload = {"model": llm.model, "messages": messages, "tools": TOOLS}
    response = requests.post(url, headers=headers, json=payload, timeout=llm.timeout)
    response.raise_for_status()
    return response.json()


def run_tool_call(call):
    name = call["function"]["name"]
    if name not in REGISTRY:
        return f"Error: unknown tool {name!r}. Available tools: {sorted(REGISTRY)}"
    try:
        arguments = json.loads(call["function"]["arguments"] or "{}")
        return str(REGISTRY[name](**arguments))
    except Exception as error:
        return f"Error: {type(error).__name__}: {error}"


def run_agent(task, max_steps=15):
    messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": task}]
    tool_calls_made, input_tokens = 0, 0

    for step in range(1, max_steps + 1):
        data = call_model(messages)
        message = data["choices"][0]["message"]
        messages.append(message)
        input_tokens += (data.get("usage") or {}).get("prompt_tokens", 0)

        calls = message.get("tool_calls") or []
        if not calls:
            return {"status": "done", "summary": message.get("content"), "steps": step,
                    "tool_calls": tool_calls_made, "input_tokens": input_tokens}

        for call in calls:
            result = run_tool_call(call)
            messages.append({"role": "tool", "tool_call_id": call["id"], "content": result})
            tool_calls_made += 1
            print(f"[{step}] {call['function']['name']}({call['function']['arguments'][:60]})")
            print(f"    {result.splitlines()[0][:70] if result else '(empty)'}")

    return {"status": "max_steps", "summary": None, "steps": max_steps,
            "tool_calls": tool_calls_made, "input_tokens": input_tokens}


def source_lines(*functions):
    """Count the non-blank lines of code in these functions."""
    return sum(
        1 for f in functions for line in inspect.getsource(f).splitlines() if line.strip()
    )


print("Lines of agent code (call_model, run_tool_call, run_agent):",
      source_lines(call_model, run_tool_call, run_agent))
print("Lines of tool code (safe_path, list_files, read_file, write_file, run_tests):",
      source_lines(safe_path, list_files, read_file, write_file, run_tests))

Lines of agent code (call_model, run_tool_call, run_agent): 34
Lines of tool code (safe_path, list_files, read_file, write_file, run_tests): 33


## 4. Let it loose

In [9]:
started = time.perf_counter()
result = run_agent("The tests in this project are failing. Find the bugs and fix them.")
elapsed = time.perf_counter() - started

print()
print("Status: ", result["status"])
print("Summary:", result["summary"])

[1] list_files({})
    ["cart.py", "test_cart.py"]


[2] run_tests({})
    FAILED


[3] read_file({"path": "cart.py"})
    def subtotal(items):
[3] read_file({"path": "test_cart.py"})
    import unittest


[4] write_file({"path": "cart.py", "content": "def subtotal(items):\n    \")
    Wrote 370 characters to cart.py


[5] run_tests({})
    PASSED



Status:  done
Summary: Fixed two bugs in cart.py: subtotal ignored quantity, and shipping charged orders of exactly 50. All tests pass now.


## 5. Don't trust it. Verify.

The model's summary is only a claim. A tool result can be misread, and a model can simply be wrong. So we check independently: **our code** runs the tests again and shows exactly what changed. This habit is what separates a real agent from a demo, and in Part 4 it grows into a proper eval suite.

In [10]:
after = run_tests()
passed = after.startswith("PASSED")

print(after)
print("What the agent changed:\n")
for line in difflib.unified_diff(
    BUGGY_CART.splitlines(), (workspace / "cart.py").read_text().splitlines(),
    "cart.py (before)", "cart.py (after)", lineterm="",
):
    print(line)

tests_untouched = (workspace / "test_cart.py").read_text() == TESTS

print("\n--- Scorecard ---")
print(f"Tests passing:      {passed}")
print(f"Tests left alone:   {tests_untouched}")
print(f"Model calls:        {result['steps']}")
print(f"Tool calls:         {result['tool_calls']}")
print(f"Input tokens:       {result['input_tokens']}" + (" (rough estimate)" if llm.offline else ""))
print(f"Time:               {elapsed:.1f}s" + (" (simulated)" if llm.offline else ""))

PASSED
....
----------------------------------------------------------------------
Ran 4 tests in 0.000s

OK

What the agent changed:

--- cart.py (before)
+++ cart.py (after)
@@ -1,11 +1,11 @@
 def subtotal(items):
     """Total price of all items, taking quantity into account."""
-    return sum(item["price"] for item in items)
+    return sum(item["price"] * item["quantity"] for item in items)
 
 
 def shipping(total):
     """Shipping costs 5, and is free for orders of 50 or more."""
-    return 0 if total > 50 else 5
+    return 0 if total >= 50 else 5
 
 
 def checkout_total(items):

--- Scorecard ---
Tests passing:      True
Tests left alone:   True
Model calls:        6
Tool calls:         6
Input tokens:       5693 (rough estimate)
Time:               3.2s (simulated)


In [11]:
# Checkpoint: with the offline script, the outcome is fully predictable.
# On a real model we report the result above but don't fail the notebook, since models vary.
if llm.offline:
    assert passed, "The scripted fix should make the tests pass."
    assert tests_untouched
    assert result["status"] == "done"
print("Checkpoint passed." if passed else "The agent didn't solve it this time. Try a stronger model or a higher max_steps.")

Checkpoint passed.


**A real example of why this matters.** While testing this notebook, a small local model (Gemma 4 on Ollama) fixed both bugs correctly, then reported that it had "fixed three issues". The diff showed two changed lines. The code was right and the summary was wrong, and only the independent check reveals that.

**Why check `tests_untouched`?** Because the laziest way to make tests pass is to change the tests. Real agents do this sometimes, especially under pressure. We forbade it in the prompt, but a prompt isn't a guarantee, so we verify it in code.

## 6. What this agent can't do yet

We have a real agent, and an honest look at its gaps is the roadmap for the rest of the course.

| Limitation | What goes wrong | Fixed in |
|---|---|---|
| `write_file` replaces whole files | It wastes tokens, and one mistake can wipe out a large file | Part 2 |
| No shell, no real sandbox | It can't install or build anything, and test code isn't contained | Part 2 |
| No confirmation before risky actions | You can't approve or reject its changes | Part 2 |
| History grows without limit | Big projects overflow the context window | Part 3 |
| No way to search the code | It can only read files one at a time | Part 3 |
| No retries or evals | One API error ends the run, and we can't tell if a change improved it | Part 4 |
| One agent, one thread | It can't hand off large side tasks | Part 5 |
| It lives in a notebook | You can't run it on your own projects | Part 6 |

## Exercises

1. Add a third bug to `BUGGY_CART`, such as a broken `checkout_total`, and add a test for it. Can your model find all three?
2. Write a `search_code(pattern)` tool, and watch whether a real model starts preferring it to reading whole files.
3. Add a failing test that **can't** pass, such as `assertEqual(1, 2)`. Does the model admit defeat, keep looping or edit the test? Your scorecard should catch that last one.
4. Delete the "don't edit them" sentence from the system prompt and rerun. What changes?

## Clean up

In [12]:
shutil.rmtree(workspace)
print("Removed the temporary workspace.")

Removed the temporary workspace.


## You finished Part 1

You started with one HTTP request, and you now have an agent that diagnoses and fixes code, built entirely by hand. Most of what a coding agent does is this loop. The rest is making it safe, efficient and reliable, and that's what the next parts cover.

**Coming next: Part 2, Tools.** Precise file edits, a shell, permissions and a real sandbox.

If this helped, a star on the repo helps other people find it.